# Unit III - Text Categorization and Clustering
## Practical Implementation for BBA Students
---

In [ ]:
!pip install nltk scikit-learn pandas matplotlib seaborn wordcloud

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

## 1. Text Categorization — Spam Detection with Naive Bayes

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import pandas as pd
import numpy as np

# Sample email dataset
emails = [
    # Spam emails
    "Congratulations! You won a free iPhone. Click here now!",
    "URGENT: You have won $1,000,000. Claim your prize today!",
    "Free gift card! Limited offer. Buy now and save big!",
    "Make money fast! Work from home and earn thousands!",
    "Discount offer! 90% off designer watches. Order today!",
    "You are selected for a special reward. Click to claim!",
    "Limited time offer free trial no credit card required!",
    "Earn extra income from home guaranteed results fast!",
    "Free membership upgrade premium exclusive access now!",
    "Amazing deal buy one get one free limited stock hurry!",
    # Not Spam (Ham)
    "Hi, can we schedule a meeting for tomorrow at 3 PM?",
    "Please find the quarterly report attached for your review.",
    "The team meeting has been rescheduled to Friday.",
    "Can you send me the project proposal by end of day?",
    "Thank you for your feedback on the presentation.",
    "Reminder: Your dentist appointment is tomorrow at 10 AM.",
    "Please review the budget document and share your thoughts.",
    "The client approved the design. Let's proceed with development.",
    "Happy birthday! Wishing you a wonderful year ahead.",
    "Don't forget to submit your timesheet by Friday."
]

labels = ['spam']*10 + ['ham']*10

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    emails, labels, test_size=0.3, random_state=42
)

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

# Train Naive Bayes classifier
classifier = MultinomialNB()
classifier.fit(X_train_tfidf, y_train)

# Predict
predictions = classifier.predict(X_test_tfidf)

print("=== SPAM DETECTION RESULTS ===")
print(f"\nAccuracy: {accuracy_score(y_test, predictions)*100:.1f}%")
print("\nDetailed Report:")
print(classification_report(y_test, predictions))

# Show predictions
print("\n=== INDIVIDUAL PREDICTIONS ===")
for email, actual, predicted in zip(X_test, y_test, predictions):
    status = '✓' if actual == predicted else '✗'
    print(f"{status} Email: \"{email[:50]}...\"")
    print(f"  Actual: {actual} | Predicted: {predicted}\n")

## 2. Confusion Matrix Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Create confusion matrix
cm = confusion_matrix(y_test, predictions, labels=['spam', 'ham'])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Spam', 'Predicted Ham'],
            yticklabels=['Actually Spam', 'Actually Ham'])
plt.title('Confusion Matrix - Spam Detection', fontsize=16, fontweight='bold')
plt.ylabel('Actual Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=100)
plt.show()

# Explain the matrix
print("\n=== UNDERSTANDING THE CONFUSION MATRIX ===")
print(f"True Positives (correctly caught spam): {cm[0][0]}")
print(f"False Negatives (missed spam): {cm[0][1]}")
print(f"False Positives (good email marked as spam): {cm[1][0]}")
print(f"True Negatives (correctly allowed good email): {cm[1][1]}")

## 3. Comparing Multiple Classifiers

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression

# Define classifiers
classifiers = {
    'Naive Bayes': MultinomialNB(),
    'SVM': LinearSVC(max_iter=10000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN (K=3)': KNeighborsClassifier(n_neighbors=3),
    'Logistic Regression': LogisticRegression(max_iter=10000)
}

print("=== COMPARING ML ALGORITHMS FOR TEXT CLASSIFICATION ===")
print(f"{'Algorithm':<25} {'Accuracy':<12} {'Notes'}")
print("-" * 65)

results = {}
for name, clf in classifiers.items():
    clf.fit(X_train_tfidf, y_train)
    preds = clf.predict(X_test_tfidf)
    acc = accuracy_score(y_test, preds)
    results[name] = acc
    
    notes = {
        'Naive Bayes': 'Fast, great for text',
        'SVM': 'Best for high-dimensional data',
        'Decision Tree': 'Easy to interpret',
        'KNN (K=3)': 'Simple, no training needed',
        'Logistic Regression': 'Good baseline'
    }
    print(f"{name:<25} {acc*100:.1f}%{'':<7} {notes[name]}")

# Visualize comparison
plt.figure(figsize=(10, 5))
bars = plt.bar(results.keys(), [v*100 for v in results.values()], 
               color=['#1565C0', '#D32F2F', '#2E7D32', '#F57F17', '#6A1B9A'])
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Classifier Comparison for Spam Detection', fontsize=14, fontweight='bold')
plt.ylim(0, 110)
for bar, val in zip(bars, results.values()):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
             f'{val*100:.0f}%', ha='center', fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig('classifier_comparison.png', dpi=100)
plt.show()

## 4. Text Clustering with K-Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# More documents for clustering
documents = [
    # Sports-related
    "The team won the championship game in overtime.",
    "The player scored three goals in the football match.",
    "Cricket world cup finals were exciting to watch.",
    "The tennis tournament had an unexpected winner.",
    # Technology-related
    "Apple released a new iPhone with better AI features.",
    "Google announced improvements to their search algorithm.",
    "The new laptop has faster processor and more RAM.",
    "Cloud computing is transforming digital business.",
    # Business-related
    "Stock market reached an all-time high today.",
    "The company reported strong quarterly earnings.",
    "New trade policies will impact global markets.",
    "Startup raised funding in latest investment round."
]

# Convert to TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(documents)

# Apply K-Means with K=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(tfidf_matrix)

print("=== K-MEANS CLUSTERING RESULTS ===")
print("Documents automatically grouped into 3 clusters:\n")

for cluster_id in range(3):
    print(f"--- Cluster {cluster_id + 1} ---")
    for i, doc in enumerate(documents):
        if clusters[i] == cluster_id:
            print(f"  • {doc}")
    
    # Top words in each cluster
    center = kmeans.cluster_centers_[cluster_id]
    top_indices = center.argsort()[-5:][::-1]
    top_words = [tfidf.get_feature_names_out()[i] for i in top_indices]
    print(f"  Top words: {', '.join(top_words)}\n")

## 5. Visualizing Clusters

In [ ]:
# Reduce dimensions for visualization
pca = PCA(n_components=2, random_state=42)
reduced = pca.fit_transform(tfidf_matrix.toarray())

colors = ['#D32F2F', '#1565C0', '#2E7D32']
cluster_names = ['Cluster 1', 'Cluster 2', 'Cluster 3']

plt.figure(figsize=(10, 7))
for i in range(3):
    mask = clusters == i
    plt.scatter(reduced[mask, 0], reduced[mask, 1], 
                c=colors[i], label=cluster_names[i], s=150, edgecolors='black', alpha=0.7)
    
    # Add labels
    for j, (x, y) in enumerate(reduced[mask]):
        doc_idx = np.where(mask)[0][j]
        short_text = documents[doc_idx][:30] + '...'
        plt.annotate(short_text, (x, y), fontsize=7, ha='center', va='bottom')

plt.xlabel('Component 1', fontsize=12)
plt.ylabel('Component 2', fontsize=12)
plt.title('Text Document Clustering (K-Means, K=3)', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('text_clustering.png', dpi=100)
plt.show()

## 6. Hierarchical Clustering with Dendrogram

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage

# Compute linkage
linkage_matrix = linkage(tfidf_matrix.toarray(), method='ward')

# Create short labels
short_labels = [doc[:35] + '...' for doc in documents]

plt.figure(figsize=(12, 7))
dendrogram(linkage_matrix, labels=short_labels, leaf_rotation=45, leaf_font_size=9)
plt.title('Hierarchical Clustering Dendrogram', fontsize=14, fontweight='bold')
plt.xlabel('Documents', fontsize=12)
plt.ylabel('Distance (dissimilarity)', fontsize=12)
plt.tight_layout()
plt.savefig('dendrogram.png', dpi=100)
plt.show()

print("\nHow to read the dendrogram:")
print("- Documents at the bottom are individual items")
print("- Lines connecting them show which documents are most similar")
print("- Lower the merge point = more similar the documents")
print("- Cut the tree horizontally at any height to get clusters")